In [4]:
!pip install numpy
!pip install pandas

Defaulting to user installation because normal site-packages is not writeable
Defaulting to user installation because normal site-packages is not writeable


In [1]:
import os
import glob
import json
from collections import defaultdict
import numpy as np
import pandas as pd

In [ ]:
import os
import glob
import json
from collections import defaultdict
import pandas as pd

RAW_BASE = "RawData"
OUT_MATCH_CSV = "data_processed/venue_match_level.csv"
OUT_VENUE_CSV = "data_processed/venue_master.csv"


# INNINGS STATS

def innings_stats(innings):
    if innings is None:
        return None

    runs = balls = 0
    wp = wm = wd = 0
    r_pp = r_mid = r_death = 0

    for over in innings.get("overs", []):
        over_no = over.get("over", 0)

        for d in over.get("deliveries", []):
            r = d.get("runs", {}).get("total", 0)

            runs += r
            balls += 1

            if over_no <= 5:
                r_pp += r
            elif over_no <= 15:
                r_mid += r
            else:
                r_death += r

            if d.get("wickets"):
                if over_no <= 5:
                    wp += 1
                elif over_no <= 15:
                    wm += 1
                else:
                    wd += 1

    overs = balls / 6 if balls else 0
    runrate = runs / overs if overs else None

    return {
        "runs": runs,
        "runrate": runrate,
        "wp": wp, "wm": wm, "wd": wd,
        "r_pp": r_pp, "r_mid": r_mid, "r_death": r_death
    }


matches = []
venue_agg = defaultdict(lambda: {
    "count": 0,
    "city": None,
    "sum_first": 0,
    "sum_second": 0,
    "sum_fr": 0,
    "sum_sr": 0,
    "fr_count": 0,
    "sr_count": 0,
    "wp_f": 0, "wm_f": 0, "wd_f": 0,
    "wp_s": 0, "wm_s": 0, "wd_s": 0,
    "rpp_f": 0, "rmid_f": 0, "rdeath_f": 0,
    "rpp_s": 0, "rmid_s": 0, "rdeath_s": 0,
})


for year in range(2014, 2026):
    path = os.path.join(RAW_BASE, str(year))
    if not os.path.exists(path):
        continue

    for file in glob.glob(f"{path}/*.json"):
        with open(file, "r", encoding="utf-8") as f:
            match = json.load(f)

        info = match.get("info", {})
        if info.get("match_type", "").upper() != "T20":
            continue

        teams = info.get("teams", [])
        if len(teams) != 2:
            continue

    
        # LEAGUE TAGGING
        
        team_type = info.get("team_type", "").lower()
        event_name = info.get("event", {}).get("name", "").lower()

        if team_type == "international":
            league = "International"
        elif "ipl" in event_name:
            league = "IPL"
        elif "syed" in event_name or "mustaq" in event_name:
            league = "SMAT"
        elif "big bash" in event_name:
            league = "BBL"
        elif "psl" in event_name:
            league = "PSL"
        else:
            league = "Domestic"

        venue = info.get("venue")
        city = info.get("city")
        match_id = os.path.basename(file).replace(".json", "")

       
        match_date = None
        if info.get("dates"):
            match_date = info["dates"][0]

        
        outcome = info.get("outcome", {})
        winner = outcome.get("winner")

        victory_type = None
        victory_margin = None

        if "by" in outcome:
            if "runs" in outcome["by"]:
                victory_type = "runs"
                victory_margin = outcome["by"]["runs"]
            elif "wickets" in outcome["by"]:
                victory_type = "wickets"
                victory_margin = outcome["by"]["wickets"]
        else:
            result = outcome.get("result")
            if result:
                victory_type = result  # tie / no result / abandoned

        innings = match.get("innings", [])
        i1 = innings_stats(innings[0]) if len(innings) > 0 else None
        i2 = innings_stats(innings[1]) if len(innings) > 1 else None

     
        # MATCH ROW
       
        row = {
            "match_id": match_id,
            "match_date": match_date,
            "league": league,
            "team1": teams[0],
            "team2": teams[1],
            "venue": venue,
            "city": city,

            "winner": winner,
            "victory_type": victory_type,
            "victory_margin": victory_margin,

            "first_innings_score": i1["runs"] if i1 else None,
            "second_innings_score": i2["runs"] if i2 else None,
            "first_innings_runrate": i1["runrate"] if i1 else None,
            "second_innings_runrate": i2["runrate"] if i2 else None,

            "first_wp": i1["wp"] if i1 else 0,
            "first_wm": i1["wm"] if i1 else 0,
            "first_wd": i1["wd"] if i1 else 0,
            "second_wp": i2["wp"] if i2 else 0,
            "second_wm": i2["wm"] if i2 else 0,
            "second_wd": i2["wd"] if i2 else 0,

            "first_runs_pp": i1["r_pp"] if i1 else 0,
            "first_runs_middle": i1["r_mid"] if i1 else 0,
            "first_runs_death": i1["r_death"] if i1 else 0,
            "second_runs_pp": i2["r_pp"] if i2 else 0,
            "second_runs_middle": i2["r_mid"] if i2 else 0,
            "second_runs_death": i2["r_death"] if i2 else 0,
        }

        matches.append(row)

        
        # VENUE AGGREGATION 
        
        v = venue_agg[venue]
        v["count"] += 1
        v["city"] = city

        if i1:
            v["sum_first"] += i1["runs"]
            if i1["runrate"] is not None:
                v["sum_fr"] += i1["runrate"]
                v["fr_count"] += 1
            v["wp_f"] += i1["wp"]
            v["wm_f"] += i1["wm"]
            v["wd_f"] += i1["wd"]
            v["rpp_f"] += i1["r_pp"]
            v["rmid_f"] += i1["r_mid"]
            v["rdeath_f"] += i1["r_death"]

        if i2:
            v["sum_second"] += i2["runs"]
            if i2["runrate"] is not None:
                v["sum_sr"] += i2["runrate"]
                v["sr_count"] += 1
            v["wp_s"] += i2["wp"]
            v["wm_s"] += i2["wm"]
            v["wd_s"] += i2["wd"]
            v["rpp_s"] += i2["r_pp"]
            v["rmid_s"] += i2["r_mid"]
            v["rdeath_s"] += i2["r_death"]


pd.DataFrame(matches).to_csv(OUT_MATCH_CSV, index=False)

venue_rows = []
for venue, v in venue_agg.items():
    venue_rows.append({
        "venue": venue,
        "city": v["city"],
        "matches_played": v["count"],
        "avg_first_innings_score": v["sum_first"] / v["count"],
        "avg_second_innings_score": v["sum_second"] / v["count"],
        "avg_first_innings_runrate": v["sum_fr"] / v["fr_count"] if v["fr_count"] else None,
        "avg_second_innings_runrate": v["sum_sr"] / v["sr_count"] if v["sr_count"] else None,
        "avg_wickets_pp_first": v["wp_f"] / v["count"],
        "avg_wickets_middle_first": v["wm_f"] / v["count"],
        "avg_wickets_death_first": v["wd_f"] / v["count"],
        "avg_wickets_pp_second": v["wp_s"] / v["count"],
        "avg_wickets_middle_second": v["wm_s"] / v["count"],
        "avg_wickets_death_second": v["wd_s"] / v["count"],
        "avg_runs_pp_first": v["rpp_f"] / v["count"],
        "avg_runs_middle_first": v["rmid_f"] / v["count"],
        "avg_runs_death_first": v["rdeath_f"] / v["count"],
        "avg_runs_pp_second": v["rpp_s"] / v["count"],
        "avg_runs_middle_second": v["rmid_s"] / v["count"],
        "avg_runs_death_second": v["rdeath_s"] / v["count"],
    })

pd.DataFrame(venue_rows).to_csv(OUT_VENUE_CSV, index=False)

print("Saved:")
print(" -", OUT_MATCH_CSV)
print(" -", OUT_VENUE_CSV)


Saved:
 - data_processed/venue_match_level.csv
 - data_processed/venue_master.csv


In [4]:
import pandas as pd

df = pd.read_csv("data_processed/venue_match_level.csv")

print("\nLEAGUE COUNTS")
print(df["league"].value_counts())

print("\nTEAM TYPE CHECK")
print(df[["team1", "team2"]].drop_duplicates().head(20))



LEAGUE COUNTS
league
Domestic         4412
International    2513
SMAT              616
BBL               540
Name: count, dtype: int64

TEAM TYPE CHECK
           team1        team2
0      Australia      England
3    West Indies      England
6   South Africa    Australia
8        Knights     Warriors
9        Knights     Dolphins
10   Cape Cobras       Titans
11   Cape Cobras     Warriors
12      Dolphins       Titans
13         Lions      Knights
14   Cape Cobras        Lions
15   Cape Cobras      Knights
16      Warriors     Dolphins
17        Titans        Lions
18      Warriors        Lions
19   Cape Cobras     Dolphins
20        Titans      Knights
23      Dolphins  Cape Cobras
24      Warriors       Titans
27        Titans     Dolphins
28      Dolphins      Knights


filtering

In [ ]:
import pandas as pd

IN_CSV = "data_processed/venue_match_level.csv"
OUT_CSV = "data_processed/venue_match_level.csv"  # overwrite
# OUT_CSV = "data_processed/venue_match_level_tagged.csv"  # optional new file

SMAT_TEAMS = {
    "Andhra",
    "Arunachal Pradesh",
    "Assam",
    "Baroda",
    "Bengal",
    "Bihar",
    "Chandigarh",
    "Chhattisgarh",
    "Delhi",
    "Goa",
    "Gujarat",
    "Haryana",
    "Himachal",
    "Himachal Pradesh",
    "Hyderabad (India)",
    "Jammu & Kashmir",
    "Jharkhand",
    "Karnataka",
    "Kerala",
    "Madhya Pradesh",
    "Maharashtra",
    "Manipur",
    "Meghalaya",
    "Mizoram",
    "Mumbai",
    "Nagaland",
    "Odisha",
    "Puducherry",
    "Punjab",
    "Railways",
    "Rajasthan",
    "Saurashtra",
    "Services",
    "Sikkim",
    "Tamil Nadu",
    "Tripura",
    "Uttar Pradesh",
    "Uttarakhand",
    "Vidarbha"
}

SA20_TEAMS = {
    "Cape Town Blitz",
    "Durban Heat",
    "Durban's Super Giants",
    "Joburg Super Kings",
    "Jozi Stars",
    "MI Cape Town",
    "Nelson Mandela Bay Giants",
    "Paarl Royals",
    "Paarl Rocks",
    "Pretoria Capitals",
    "Sunrisers Eastern Cape",
    "Tshwane Spartans",
}

BBL_TEAMS = {
    "Adelaide Strikers",
    "Brisbane Heat",
    "Hobart Hurricanes",
    "Melbourne Renegades",
    "Melbourne Stars",
    "Perth Scorchers",
    "Sydney Sixers",
    "Sydney Thunder"
}

IPL_TEAMS = {
    "Chennai Super Kings",
    "Deccan Chargers",
    "Delhi Capitals",
    "Delhi Daredevils",
    "Gujarat Lions",
    "Gujarat Titans",
    "Kings XI Punjab",
    "Kochi Tuskers Kerala",
    "Kolkata Knight Riders",
    "Lucknow Super Giants",
    "Mumbai Indians",
    "Pune Warriors India",
    "Punjab Kings",
    "Rajasthan Royals",
    "Rising Pune Supergiant",
    "Rising Pune Supergiants",
    "Royal Challengers Bangalore",
    "Royal Challengers Bengaluru",
    "Sunrisers Hyderabad"
}

INTERNATIONAL_TEAMS = {
    "Afghanistan",
    "Australia",
    "Bangladesh",
    "England",
    "India",
    "Ireland",
    "New Zealand",
    "Pakistan",
    "South Africa",
    "Sri Lanka",
    "West Indies",
    "Zimbabwe",
}

BPL_TEAMS = {
    "Barisal Burners",
    "Barisal Bulls",
    "Chattogram Challengers",
    "Chattogram Royals",
    "Chittagong Kings",
    "Chittagong Vikings",
    "Comilla Victorians",
    "Cumilla Warriors",
    "Dhaka Capitals",
    "Dhaka Capital",
    "Dhaka Dominators",
    "Dhaka Dynamites",
    "Dhaka Gladiators",
    "Dhaka Platoon",
    "Durbar Rajshahi",
    "Durdanto Dhaka",
    "Duronto Rajshahi",
    "Fortune Barishal",
    "Khulna Royal Bengals",
    "Khulna Tigers",
    "Khulna Titans",
    "Minister Dhaka",
    "Noakhali Express",
    "Rajshahi Kings",
    "Rajshahi Royals",
    "Rajshahi Warriors",
    "Rangpur Rangers",
    "Rangpur Riders",
    "Sylhet Sixers",
    "Sylhet Strikers",
    "Sylhet Sunrisers",
    "Sylhet Super Stars",
    "Sylhet Thunder",
    "Sylhet Titans",
    "Sylhet Royals",
    "Minister Group Dhaka"
}

PSL_TEAMS ={
        "Islamabad United",
    "Karachi Kings",
    "Lahore Qalandars",
    "Multan Sultans",
    "Peshawar Zalmi",
    "Quetta Gladiators"
}

LISTA_SA = {
      "Boland",
    "Border",
    "Dolphins",
    "Cape Cobras",
    "Eastern Province",
    "Easterns",
    "Free State",
    "Gauteng",
    "Griqualand West",
    "Kei",
    "Knights",
    "KwaZulu-Natal",
    "KwaZulu-Natal Inland",
    "Limpopo",
    "Lions",
    "Mpumalanga",
    "Natal",
    "North West",
    "Northern Cape",
    "Northern Transvaal",
    "Northerns",
    "Orange Free State",
    "South Western Districts",
    "Titans",
    "Transvaal",
    "Warriors",
    "Western Province"
}

LISTA_NZ ={
    "Auckland Aces",
    "Auckland",
    "Canterbury",
    "Central Districts",
    "Northern Districts",
    "Otago Volts",
    "Otago",
    "Wellington Firebirds",
    "Wellington",
    "Nelson",
    "Southland",
    "New Zealand Army",
    "New Zealand Services"
}

LISTA_ENG ={
    "Bedfordshire",
    "Berkshire",
    "Buckinghamshire",
    "Cambridgeshire",
    "Cheshire",
    "Cornwall",
    "Cumbria",
    "Derbyshire",
    "Devon",
    "Durham",
    "Essex",
    "Glamorgan",
    "Gloucestershire",
    "Hampshire",
    "Herefordshire",
    "Hertfordshire",
    "Kent",
    "Lancashire",
    "Leicestershire",
    "Lincolnshire",
    "London County",
    "London Spirit",
    "Middlesex",
    "Norfolk",
    "Northamptonshire",
    "Northumberland",
    "Nottinghamshire",
    "Oxfordshire",
    "Shropshire",
    "Somerset",
    "Southern Brave",
    "Southern Counties",
    "Staffordshire",
    "Suffolk",
    "Surrey",
    "Sussex",
    "Trent Rockets",
    "Warwickshire",
    "Welsh Fire",
    "Worcestershire",
    "Yorkshire",
    "Cambridge University",
    "Oxford University",
    "Durham MCCU",
    "Loughborough MCCU",
    "Cardiff MCCU",
    "Leeds/Bradford MCCU",
    "Marylebone Cricket Club (MCC)",
    "Unicorns",
      "Birmingham Phoenix",
    "London Spirit",
    "Manchester Originals",
    "Northern Superchargers",
    "Oval Invincibles",
    "Southern Brave",
    "Trent Rockets",
    "Welsh Fire",
    "Birmingham Bears"
}

LPL_teams = {
    "B-Love Kandy",
    "Colombo Kings",
    "Colombo Stars",
    "Colombo Strikers",
    "Dambulla Aura",
    "Dambulla Sixers",
    "Dambulla Giants",
    "Dambulla Viiking",
    "Galle Gladiators",
    "Galle Marvels",
    "Galle Titans",
    "Jaffna Kings",
    "Jaffna Stallions",
    "Kandy Falcons",
    "Kandy Tuskers",
    "Kandy Warriors"
}

mlc_teams = {
    "Los Angeles Knight Riders",
    "MI New York",
    "San Francisco Unicorns",
    "Seattle Orcas",
    "Texas Super Kings",
    "Washington Freedom"
}

# International League T20 (ILT20) teams (current)
ilt20_teams = {
    "Abu Dhabi Knight Riders",
    "Desert Vipers",
    "Dubai Capitals",
    "Gulf Giants",
    "MI Emirates",
    "Sharjah Warriors",
    "Sharjah Warriorz"
}

CPL_TEAMS ={
     "Antigua & Barbuda Falcons",
     "Antigua and Barbuda Falcons",
    "Antigua Hawksbills",
    "Barbados Royals",
    "Barbados Tridents",
    "Guyana Amazon Warriors",
    "Jamaica Tallawahs",
    "St Kitts and Nevis Patriots",
    "St Lucia Kings",
    "St Lucia Stars",
    "St Lucia Zouks",
    "Trinidad & Tobago Red Steel",
    "Trinidad and Tobago Red Steel",
    "Trinbago Knight Riders"
}


SMAT_TEAMS = {t.lower() for t in SMAT_TEAMS}
BBL_TEAMS = {t.lower() for t in BBL_TEAMS}
IPL_TEAMS = {t.lower() for t in IPL_TEAMS}
INTERNATIONAL_TEAMS = {t.lower() for t in INTERNATIONAL_TEAMS}
BPL_TEAMS = {t.lower() for t in BPL_TEAMS}
SA20_TEAMS = {t.lower() for t in SA20_TEAMS}
PSL_TEAMS = {t.lower() for t in PSL_TEAMS}
LISTA_SA = {t.lower() for t in  LISTA_SA}
LISTA_NZ = {t.lower() for t in LISTA_NZ}
LISTA_ENG = {t.lower() for t in LISTA_ENG}
LPL_teams = {t.lower() for t in LPL_teams}
mlc_teams = {t.lower() for t in mlc_teams}
ilt20_teams = {t.lower() for t in ilt20_teams}
CPL_TEAMS ={t.lower() for t in CPL_TEAMS}


df = pd.read_csv(IN_CSV)

def classify_league(row):
    t1 = row["team1"].strip().lower()
    t2 = row["team2"].strip().lower()

    # International
    if t1 in INTERNATIONAL_TEAMS and t2 in INTERNATIONAL_TEAMS:
        return "International"

    # Franchise / Domestic
    if t1 in IPL_TEAMS and t2 in IPL_TEAMS:
        return "IPL"

    if t1 in SMAT_TEAMS and t2 in SMAT_TEAMS:
        return "SMAT"

    if t1 in BBL_TEAMS and t2 in BBL_TEAMS:
        return "BBL"
    
    if t1 in BPL_TEAMS and t2 in BPL_TEAMS:
        return "BPL"
    
    if t1 in SA20_TEAMS and t2 in SA20_TEAMS:
        return "SA20"
    
    if t1 in PSL_TEAMS and t2 in PSL_TEAMS:
        return "PSL"

    if t1 in LISTA_SA and t2 in LISTA_SA:
        return "LISTA_SA"
    
    if t1 in LISTA_NZ and t2 in LISTA_NZ:
        return "LISTA_NZ"
    
    if t1 in LISTA_ENG and t2 in LISTA_ENG:
        return "LISTA_ENG"
    
    if t1 in LPL_teams and t2 in LPL_teams:
        return "LPL"
    
    if t1 in mlc_teams and t2 in mlc_teams:
        return "MLC"
    
    if t1 in ilt20_teams and t2 in ilt20_teams:
        return "ILT20"
    
    if t1 in CPL_TEAMS and t2 in CPL_TEAMS:
        return "CPL"
    
    return "UNKNOWN"


df["league"] = df.apply(classify_league, axis=1)


df.to_csv(OUT_CSV, index=False)

print("Saved:", OUT_CSV)
print("\nLeague distribution:")
print(df["league"].value_counts())

print("\nSample UNKNOWN rows:")
print(df[df["league"] == "UNKNOWN"][["team1", "team2"]].drop_duplicates().head(15))


Saved: data_processed/venue_match_level.csv

League distribution:
league
UNKNOWN          2026
LISTA_ENG        1581
IPL               711
International     689
SMAT              616
BBL               540
BPL               380
CPL               319
PSL               309
LISTA_SA          259
LISTA_NZ          232
SA20              125
LPL               119
ILT20             100
MLC                75
Name: count, dtype: int64

Sample UNKNOWN rows:
                    team1                 team2
66              Hong Kong                 Nepal
68            Netherlands  United Arab Emirates
69             Bangladesh                 Nepal
70            Netherlands              Zimbabwe
71                Ireland  United Arab Emirates
72             Bangladesh             Hong Kong
73   United Arab Emirates              Zimbabwe
74                Ireland           Netherlands
81            Netherlands             Sri Lanka
83            Netherlands          South Africa
87            Netherl

In [2]:
import pandas as pd

IN_CSV = "data_processed/venue_match_level.csv"
OUT_CSV = "data_processed/venue_match_level_clean.csv"

df = pd.read_csv(IN_CSV)

df_clean = df[df["league"] != "UNKNOWN"]

df_clean.to_csv(OUT_CSV, index=False)

print("Saved:", OUT_CSV)
print("\nLeague distribution:")
print(df_clean["league"].value_counts())


Saved: data_processed/venue_match_level_clean.csv

League distribution:
league
LISTA_ENG        1581
IPL               711
International     689
SMAT              616
BBL               540
BPL               380
CPL               319
PSL               309
LISTA_SA          259
LISTA_NZ          232
SA20              125
LPL               119
ILT20             100
MLC                75
Name: count, dtype: int64


In [ ]:
import pandas as pd

IN_CSV = "data_processed/venue_match_level_clean.csv"


df = pd.read_csv(IN_CSV)


unknown_df = df[df["league"] == "UNKNOWN"]


teams_1 = unknown_df["team1"].dropna().str.strip()
teams_2 = unknown_df["team2"].dropna().str.strip()

distinct_teams = sorted(set(teams_1).union(set(teams_2)))


print(f"\nDistinct UNKNOWN teams ({len(distinct_teams)}):\n")

for team in distinct_teams:
    print(team)



Distinct UNKNOWN teams (0):



master merged code

In [ ]:
import pandas as pd


VENUE_CSV = "data_processed/venue_match_level_clean.csv"
CLIMATE_CSV = "data_processed/climate_match_cache.csv"

OUT_CSV = "data_processed/match_master_clean_mergedwithclimate.csv"


venue_df = pd.read_csv(VENUE_CSV)
climate_df = pd.read_csv(CLIMATE_CSV)


venue_df["match_id"] = venue_df["match_id"].astype(str)
climate_df["match_id"] = climate_df["match_id"].astype(str)


merged_df = venue_df.merge(
    climate_df,
    on="match_id",
    how="left"
)


merged_df.to_csv(OUT_CSV, index=False)


print("Saved:", OUT_CSV)
print("\nRow count check:")
print("Venue base rows :", len(venue_df))
print("Merged rows     :", len(merged_df))

print("\nMissing values after merge:")
print(merged_df.isna().sum()[merged_df.isna().sum() > 0])


Saved: data_processed/match_master_clean_mergedwithclimate.csv

Row count check:
Venue base rows : 6055
Merged rows     : 6055

Missing values after merge:
city                       718
winner                     199
victory_type                 1
victory_margin             200
second_innings_score        98
second_innings_runrate      98
latitude                    16
longitude                   16
match_date_y                16
tavg                       647
tmax                       771
tmin                       811
rhum                      6055
prcp                      2149
wspd                       739
pres                      1450
tsun                      5511
dtype: int64


aggregate venue data

In [ ]:
import pandas as pd


MATCH_CSV = "data_processed/venue_match_level_clean.csv"
VENUE_OUT = "data_processed/venue_master.csv"


df = pd.read_csv(MATCH_CSV)


grouped = df.groupby(["venue", "city"], dropna=False)

venue_master = grouped.agg(
    matches_played=("match_id", "count"),

    avg_first_innings_score=("first_innings_score", "mean"),
    avg_second_innings_score=("second_innings_score", "mean"),

    avg_first_innings_runrate=("first_innings_runrate", "mean"),
    avg_second_innings_runrate=("second_innings_runrate", "mean"),

    avg_wickets_pp_first=("first_wp", "mean"),
    avg_wickets_middle_first=("first_wm", "mean"),
    avg_wickets_death_first=("first_wd", "mean"),

    avg_wickets_pp_second=("second_wp", "mean"),
    avg_wickets_middle_second=("second_wm", "mean"),
    avg_wickets_death_second=("second_wd", "mean"),

    avg_runs_pp_first=("first_runs_pp", "mean"),
    avg_runs_middle_first=("first_runs_middle", "mean"),
    avg_runs_death_first=("first_runs_death", "mean"),

    avg_runs_pp_second=("second_runs_pp", "mean"),
    avg_runs_middle_second=("second_runs_middle", "mean"),
    avg_runs_death_second=("second_runs_death", "mean"),
).reset_index()


venue_master.to_csv(VENUE_OUT, index=False)

print("Saved:", VENUE_OUT)
print("\nTotal venues:", len(venue_master))
print("\nSample rows:")
print(venue_master.head())


Saved: data_processed/venue_master.csv

Total venues: 333

Sample rows:
                                          venue       city  matches_played  \
0   Emerald Heights International School Ground        NaN              22   
1                        ACA Stadium, Barsapara        NaN              10   
2                       ACA Stadium,Mangalagiri        NaN               7   
3           Abhimanyu Cricket Academy, Dehradun  Dehra Dun              10   
4                                 Adelaide Oval   Adelaide              58   

   avg_first_innings_score  avg_second_innings_score  \
0               151.000000                132.727273   
1               144.100000                127.600000   
2               162.142857                103.571429   
3               150.200000                119.800000   
4               172.086207                153.775862   

   avg_first_innings_runrate  avg_second_innings_runrate  \
0                   7.393244                    7.189479   
1 

grouping venues with same names

In [ ]:
import pandas as pd


MATCH_CSV = "data_processed/venue_match_level_clean.csv"
CLIMATE_CSV = "data_processed/climate_match_cache.csv"
LOOKUP_CSV = "data_processed/venue_lookup.csv"
OUT_VENUE_CSV = "data_processed/venue_master.csv"


matches = pd.read_csv(MATCH_CSV)
climate = pd.read_csv(CLIMATE_CSV)
lookup = pd.read_csv(LOOKUP_CSV)


matches["match_id"] = matches["match_id"].astype(str)
climate["match_id"] = climate["match_id"].astype(str)


df = matches.merge(
    climate[["match_id", "latitude", "longitude"]],
    on="match_id",
    how="left"
)


df["latitude"] = pd.to_numeric(df["latitude"], errors="coerce")
df["longitude"] = pd.to_numeric(df["longitude"], errors="coerce")

lookup["latitude"] = pd.to_numeric(lookup["latitude"], errors="coerce")
lookup["longitude"] = pd.to_numeric(lookup["longitude"], errors="coerce")

df["lat_round"] = df["latitude"].round(3)
df["lon_round"] = df["longitude"].round(3)

lookup["lat_round"] = lookup["latitude"].round(3)
lookup["lon_round"] = lookup["longitude"].round(3)


df = df.merge(
    lookup[
        [
            "venue_id",
            "canonical_name",
            "city",
            "latitude",
            "longitude",
            "lat_round",
            "lon_round",
        ]
    ],
    on=["lat_round", "lon_round"],
    how="left",
    suffixes=("", "_lookup"),
)


df = df[df["venue_id"].notna()]


grouped = df.groupby("venue_id")


venue_master = grouped.agg(
    canonical_name=("canonical_name", "first"),
    city=("city", "first"),

    latitude=("latitude_lookup", "mean"),
    longitude=("longitude_lookup", "mean"),

    matches_played=("match_id", "count"),

    avg_first_innings_score=("first_innings_score", "mean"),
    avg_second_innings_score=("second_innings_score", "mean"),

    avg_first_innings_runrate=("first_innings_runrate", "mean"),
    avg_second_innings_runrate=("second_innings_runrate", "mean"),

    avg_wickets_pp_first=("first_wp", "mean"),
    avg_wickets_middle_first=("first_wm", "mean"),
    avg_wickets_death_first=("first_wd", "mean"),

    avg_wickets_pp_second=("second_wp", "mean"),
    avg_wickets_middle_second=("second_wm", "mean"),
    avg_wickets_death_second=("second_wd", "mean"),

    avg_runs_pp_first=("first_runs_pp", "mean"),
    avg_runs_middle_first=("first_runs_middle", "mean"),
    avg_runs_death_first=("first_runs_death", "mean"),

    avg_runs_pp_second=("second_runs_pp", "mean"),
    avg_runs_middle_second=("second_runs_middle", "mean"),
    avg_runs_death_second=("second_runs_death", "mean"),
).reset_index()


venue_master.to_csv(OUT_VENUE_CSV, index=False)


print("Saved:", OUT_VENUE_CSV)
print("Unique venue_ids:", venue_master["venue_id"].nunique())
print("Rows in venue_master:", len(venue_master))


Saved: data_processed/venue_master.csv
Unique venue_ids: 162
Rows in venue_master: 162


manual merge

In [ ]:
import pandas as pd

VENUE_CSV = "data_processed/venue_master.csv"
OUT_CSV = "data_processed/venue_master.csv"

df = pd.read_csv(VENUE_CSV)


VENUE_MERGE_MAP = {
    "BARSAPARA_CRICKET_STADIUM_GUWAHATI": "BARSAPARA_CRICKET_STADIUM",
    "BREADY_CRICKET_CLUB_MAGHERAMASON": "BREADY_CRICKET_CLUB_MAGHERAMASON_BREADY",
    "BRIAN_LARA_STADIUM_TAROUBA": "BRIAN_LARA_STADIUM_TAROUBA_TRINIDAD",
    "COLLEGE_GROUND": "COLLEGE_GROUND_CHELTENHAM",
    "COUNTY_GROUND_NEW_ROAD": "COUNTY_GROUND_NEW_ROAD_WORCESTER",
    "DAREN_SAMMY_NATIONAL_CRICKET_STADIUM_GROS_ISLET": "DAREN_SAMMY_NATIONAL_CRICKET_STADIUM_GROS_ISLET_ST_LUCIA",
    "EDEN_GARDENS": "EDEN_GARDENS_KOLKATA",
    "INTERNATIONAL_SPORTS_STADIUM": "INTERNATIONAL_SPORTS_STADIUM_COFFS_HARBOUR",
    "NATIONAL_STADIUM": "NATIONAL_STADIUM_KARACHI",
    "QUEEN_S_PARK_OVAL_PORT_OF_SPAIN": "QUEEN_S_PARK_OVAL_PORT_OF_SPAIN_TRINIDAD",
    "SHERE_BANGLA_NATIONAL_STADIUM": "SHERE_BANGLA_NATIONAL_STADIUM_MIRPUR",
    "SIMONDS_STADIUM_SOUTH_GEELONG": "SIMONDS_STADIUM_SOUTH_GEELONG_VICTORIA",
    "SPORTPARK_WESTVLIET_THE_HAGUE": "SPORTPARK_WESTVLIET_VOORBURG",
}

df["venue_id"] = df["venue_id"].replace(VENUE_MERGE_MAP)


def wmean(x, w):
    return (x * w).sum() / w.sum()


# GROUP + WEIGHTED AGG

rows = []

for vid, g in df.groupby("venue_id"):
    w = g["matches_played"]

    rows.append({
        "venue_id": vid,
        "canonical_name": g["canonical_name"].iloc[0],
        "city": g["city"].iloc[0],
        "latitude": g["latitude"].mean(),
        "longitude": g["longitude"].mean(),

        "matches_played": w.sum(),

        "avg_first_innings_score": wmean(g["avg_first_innings_score"], w),
        "avg_second_innings_score": wmean(g["avg_second_innings_score"], w),

        "avg_first_innings_runrate": wmean(g["avg_first_innings_runrate"], w),
        "avg_second_innings_runrate": wmean(g["avg_second_innings_runrate"], w),

        "avg_wickets_pp_first": wmean(g["avg_wickets_pp_first"], w),
        "avg_wickets_middle_first": wmean(g["avg_wickets_middle_first"], w),
        "avg_wickets_death_first": wmean(g["avg_wickets_death_first"], w),

        "avg_wickets_pp_second": wmean(g["avg_wickets_pp_second"], w),
        "avg_wickets_middle_second": wmean(g["avg_wickets_middle_second"], w),
        "avg_wickets_death_second": wmean(g["avg_wickets_death_second"], w),

        "avg_runs_pp_first": wmean(g["avg_runs_pp_first"], w),
        "avg_runs_middle_first": wmean(g["avg_runs_middle_first"], w),
        "avg_runs_death_first": wmean(g["avg_runs_death_first"], w),

        "avg_runs_pp_second": wmean(g["avg_runs_pp_second"], w),
        "avg_runs_middle_second": wmean(g["avg_runs_middle_second"], w),
        "avg_runs_death_second": wmean(g["avg_runs_death_second"], w),
    })

venue_master = pd.DataFrame(rows)


venue_master.to_csv(OUT_CSV, index=False)

print("Merged venues with WEIGHTED means saved:", OUT_CSV)
print("Rows:", len(venue_master))


Merged venues with WEIGHTED means saved: data_processed/venue_master.csv
Rows: 150


adding more match context features

In [ ]:
import os
import json
import pandas as pd


RAW_ALL = "RawData/all"
MATCH_CSV = "data_processed/venue_match_level_clean.csv"
OUT_CSV = "data_processed/venue_match_level_clean.csv"  # overwrite


df = pd.read_csv(MATCH_CSV)
df["match_id"] = df["match_id"].astype(str)


bat_first = {}
bat_second = {}
winner_map = {}
win_type_map = {}
win_margin_map = {}


needed_ids = set(df["match_id"])

for fname in os.listdir(RAW_ALL):
    if not fname.endswith(".json"):
        continue

    match_id = fname.replace(".json", "")
    if match_id not in needed_ids:
        continue

    with open(os.path.join(RAW_ALL, fname), "r", encoding="utf-8") as f:
        match = json.load(f)

    info = match.get("info", {})
    innings = match.get("innings", [])


    # Batting order
 
    bat_first[match_id] = innings[0].get("team") if len(innings) > 0 else None
    bat_second[match_id] = innings[1].get("team") if len(innings) > 1 else None


    # Match result

    outcome = info.get("outcome", {})
    winner = outcome.get("winner")
    by = outcome.get("by", {})

    if "runs" in by:
        win_type_map[match_id] = "runs"
        win_margin_map[match_id] = by.get("runs")
    elif "wickets" in by:
        win_type_map[match_id] = "wickets"
        win_margin_map[match_id] = by.get("wickets")
    else:
        win_type_map[match_id] = "none"
        win_margin_map[match_id] = None

    winner_map[match_id] = winner


df["team_batted_first"] = df["match_id"].map(bat_first)
df["team_batted_second"] = df["match_id"].map(bat_second)
df["winner"] = df["match_id"].map(winner_map)
df["win_type"] = df["match_id"].map(win_type_map)
df["win_margin"] = df["match_id"].map(win_margin_map)


df.to_csv(OUT_CSV, index=False)


print("Updated and saved:", OUT_CSV)
print("\nMissing batting order:",
      df["team_batted_first"].isna().sum())

print("\nWin type distribution:")
print(df["win_type"].value_counts())


Updated and saved: data_processed/venue_match_level_clean.csv

Missing batting order: 0

Win type distribution:
win_type
wickets    2933
runs       2742
none        195
Name: count, dtype: int64


adding match date

In [ ]:
import pandas as pd


MATCH_CSV = "data_processed/venue_match_level_clean.csv"
META_CSV = "team_data_processed/t20_match_metadata_filtered.csv"
OUT_CSV = "data_processed/venue_match_level_clean.csv"  


match_df = pd.read_csv(MATCH_CSV)
meta_df = pd.read_csv(META_CSV)

match_df["match_id"] = match_df["match_id"].astype(str)
meta_df["match_id"] = meta_df["match_id"].astype(str)


match_df = match_df.merge(
    meta_df[["match_id", "date"]],
    on="match_id",
    how="left"
)


match_df.to_csv(OUT_CSV, index=False)

print("Merged date into:", OUT_CSV)
print("Missing dates:", match_df["date"].isna().sum())


Merged date into: data_processed/venue_match_level_clean.csv
Missing dates: 0


checking if merged match dates match

In [ ]:
import pandas as pd


CSV_PATH = "data_processed/match_master_clean_mergedwithclimate.csv"


df = pd.read_csv(CSV_PATH)


comparison = df["match_date_x"] == df["match_date_y"]

all_equal = comparison.all()
mismatch_count = (~comparison).sum()

print("All match_date_x == match_date_y :", all_equal)
print("Number of mismatches             :", mismatch_count)


if mismatch_count > 0:
    print("\nSample mismatched rows:")
    print(
        df.loc[~comparison, ["match_id", "match_date_x", "match_date_y"]]
        .head(10)
    )
else:
    print("\nNo mismatches found.")


All match_date_x == match_date_y : False
Number of mismatches             : 16

Sample mismatched rows:
      match_id match_date_x match_date_y
3508   1280201   2021-11-04          NaN
3509   1280202   2021-11-04          NaN
3523   1280220   2021-11-05          NaN
3524   1280221   2021-11-05          NaN
3541   1280239   2021-11-06          NaN
3542   1280240   2021-11-06          NaN
3555   1280258   2021-11-08          NaN
3556   1280259   2021-11-08          NaN
3569   1280277   2021-11-09          NaN
3570   1280278   2021-11-09          NaN


In [ ]:
import pandas as pd


CSV_PATH = "data_processed/match_master_clean_mergedwithclimate.csv"
OUT_PATH = "data_processed/match_master_clean_mergedwithclimate.csv"


df = pd.read_csv(CSV_PATH)


df = df.drop(columns=["match_date_y"], errors="ignore")
df = df.rename(columns={"match_date_x": "date"})


df.to_csv(OUT_PATH, index=False)

print("Updated file saved:", OUT_PATH)
print("Current date column present:", "date" in df.columns)


Updated file saved: data_processed/match_master_clean_mergedwithclimate.csv
Current date column present: True


In [ ]:
import pandas as pd


IN_CSV = "data_processed/match_master_clean_mergedwithclimate.csv"
OUT_CSV = "data_processed/match_master_clean_mergedwithclimate_teamnormalized.csv"


df = pd.read_csv(IN_CSV)


TEAM_RENAME_MAP = {
    # Indian Premier League (IPL)
    "royal challengers bangalore": "royal challengers bengaluru",
    "royal challengers bengaluru": "royal challengers bengaluru",
    "delhi daredevils": "delhi capitals",
    "delhi capitals": "delhi capitals",
    "kings xi punjab": "punjab kings",
    "punjab kings": "punjab kings",
    "deccan chargers": "sunrisers hyderabad",
    "sunrisers hyderabad": "sunrisers hyderabad",
    "rising pune supergiant": "rising pune supergiant",
    "rising pune supergiants": "rising pune supergiant",

    # Bangladesh Premier League (BPL)
    "chittagong kings": "chattogram royals",
    "chittagong vikings": "chattogram royals",
    "chattogram challengers": "chattogram royals",
    "chattogram royals": "chattogram royals",
    "dhaka gladiators": "dhaka capitals",
    "dhaka dynamites": "dhaka capitals",
    "dhaka platoon": "dhaka capitals",
    "minister dhaka": "dhaka capitals",
    "dhaka dominators": "dhaka capitals",
    "durdanto dhaka": "dhaka capitals",
    "dhaka capitals": "dhaka capitals",
    "sylhet royals": "sylhet titans",
    "sylhet super stars": "sylhet titans",
    "sylhet sixers": "sylhet titans",
    "sylhet thunder": "sylhet titans",
    "sylhet sunrisers": "sylhet titans",
    "sylhet strikers": "sylhet titans",
    "sylhet titans": "sylhet titans",
    "barisal burners": "fortune barishal",
    "barisal bulls": "fortune barishal",
    "fortune barishal": "fortune barishal",
    "duronto rajshahi": "rajshahi warriors",
    "rajshahi kings": "rajshahi warriors",
    "rajshahi royals": "rajshahi warriors",
    "durbar rajshahi": "rajshahi warriors",
    "rajshahi warriors": "rajshahi warriors",
    "comilla victorians": "comilla victorians",
    "cumilla warriors": "comilla victorians",

    # Caribbean Premier League (CPL)
    "barbados tridents": "barbados royals",
    "barbados royals": "barbados royals",
    "trinidad and tobago red steel": "trinbago knight riders",
    "trinbago knight riders": "trinbago knight riders",
    "st lucia zouks": "st. lucia kings",
    "st lucia stars": "st. lucia kings",
    "st. lucia kings": "st. lucia kings",

    # The Hundred
    "oval invincibles": "mi london",
    "mi london": "mi london",
    "manchester originals": "manchester super giants",
    "manchester super giants": "manchester super giants",

    # Misc cleanup
    "barbados royals (renamed)": "barbados royals",
    "trinbago knight rider (renamed)": "trinbago knight riders",
    "st lucia kings (renamed)": "st. lucia kings",

    # lpl
    "colombo kings": "colombo stars",
    "colombo stars": "colombo strikers",
    "colombo strikers": "colombo strikers",
    "dambulla aura": "dambulla aura",
    "dambulla giants": "dambulla aura",
    "galle gladiators": "galle titans",
    "galle titans": "galle titans",
    "jaffna stallions": "jaffna kings",
    "jaffna kings": "jaffna kings",
    "kandy tuskers": "kandy falcons",
    "kandy falcons": "b-love kandy",
    "b-love kandy": "b-love kandy",

}


def normalize_team(name):
    if pd.isna(name):
        return name
    key = name.strip().lower()
    return TEAM_RENAME_MAP.get(key, name)


for col in ["team1", "team2", "winner"]:
    if col in df.columns:
        df[col] = df[col].apply(normalize_team)


df.to_csv(OUT_CSV, index=False)


print("Saved normalized file:", OUT_CSV)
print("\nUnique teams (sample):")
print(sorted(set(df["team1"].dropna().str.lower()))[:20])


Saved normalized file: data_processed/match_master_clean_mergedwithclimate_teamnormalized.csv

Unique teams (sample):
['abu dhabi knight riders', 'adelaide strikers', 'andhra', 'antigua and barbuda falcons', 'antigua hawksbills', 'arunachal pradesh', 'assam', 'auckland', 'australia', 'b-love kandy', 'bangladesh', 'barbados royals', 'baroda', 'bengal', 'bihar', 'birmingham bears', 'birmingham phoenix', 'boland', 'brisbane heat', 'canterbury']


capialzing all the team name

In [ ]:
import pandas as pd


IN_CSV = "data_processed/match_master_clean_mergedwithclimate_teamnormalized.csv"
OUT_CSV = "data_processed/match_master_clean_mergedwithclimate_teamnormalized.csv"


df = pd.read_csv(IN_CSV)


def title_case(name):
    if pd.isna(name):
        return name
    return name.strip().title()


for col in ["team1", "team2", "winner"]:
    if col in df.columns:
        df[col] = df[col].apply(title_case)


df.to_csv(OUT_CSV, index=False)


print("Saved capitalized file:", OUT_CSV)
print("\nSample teams:")
print(df[["team1", "team2"]].head())


Saved capitalized file: data_processed/match_master_clean_mergedwithclimate_teamnormalized.csv

Sample teams:
         team1    team2
0    Australia  England
1    Australia  England
2    Australia  England
3  West Indies  England
4  West Indies  England
